# KM-Vector Phase 0 — 데이터 탐색 & 행렬 시각화

이 노트북은 Phase 0의 탐색 작업을 담당합니다.

1. 데이터 로드 확인
2. F 행렬 (처방-본초) 히트맵
3. S 행렬 (처방-증상) 히트맵
4. 처방 간 코사인 유사도 (F, S 각각)
5. 증상 쿼리 → 처방 추천 테스트

> ⚠️ **참고용**: 모든 출력 결과는 연구자가 임상적으로 검증해야 합니다.

In [ ]:
import sys
from pathlib import Path

# src 경로 추가
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정 (Windows)
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from load_data import load_all, expand_composition, expand_symptoms
from build_matrices import (
    build_formula_herb_matrix,
    build_formula_symptom_matrix,
    cosine_similarity_matrix,
    query_similarity,
    build_symptom_name_index,
)

print('모듈 로드 완료')

## 1. 데이터 로드

In [ ]:
data = load_all()
formulas_df  = data['formulas']
herbs_df     = data['herbs']
syndromes_df = data['syndromes']

print(f'처방: {len(formulas_df)}개 | 본초: {len(herbs_df)}종 | 변증: {len(syndromes_df)}개')

In [ ]:
# 처방 목록
formulas_df[['name_kr', 'source_clause', 'total_dose_g']]

In [ ]:
# 본초 목록
herbs_df[['name_kr', 'category']]

In [ ]:
# 처방-본초 구성 (long form)
comp = expand_composition(formulas_df)

# 처방 이름 붙이기
fname_map = formulas_df['name_kr'].to_dict()
comp['formula_name'] = comp['formula_id'].map(fname_map)
comp[['formula_name', 'name_kr', 'role', 'dose_g', 'dose_ratio']]

## 2. F 행렬 — 처방-본초 히트맵

In [ ]:
F = build_formula_herb_matrix(formulas_df)
print('F 행렬 shape:', F.shape)

# 표시용 이름 변환
herb_names   = herbs_df['name_kr'].to_dict()
formula_names = formulas_df['name_kr'].to_dict()

F_display = F.rename(index=formula_names, columns=herb_names)
F_display.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

im = ax.imshow(F_display.values, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='용량 비율')

ax.set_xticks(range(len(F_display.columns)))
ax.set_xticklabels(F_display.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(F_display.index)))
ax.set_yticklabels(F_display.index, fontsize=9)

ax.set_title('Formula-Herb 행렬 F (용량 비율)', fontsize=13, pad=12)
ax.set_xlabel('본초')
ax.set_ylabel('처방')

plt.tight_layout()
plt.savefig('../data/F_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: data/F_heatmap.png')

## 3. S 행렬 — 처방-증상 히트맵

In [ ]:
S = build_formula_symptom_matrix(formulas_df, syndromes_df)
print('S 행렬 shape:', S.shape)

# symptom_id → 이름 변환
symp_df = expand_symptoms(syndromes_df)
sid_to_name = symp_df.drop_duplicates('symptom_id').set_index('symptom_id')['name_kr'].to_dict()

S_display = S.rename(index=formula_names, columns=sid_to_name)
S_display.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

im = ax.imshow(S_display.values, aspect='auto', cmap='Blues')
plt.colorbar(im, ax=ax, label='가중치')

ax.set_xticks(range(len(S_display.columns)))
ax.set_xticklabels(S_display.columns, rotation=60, ha='right', fontsize=8)
ax.set_yticks(range(len(S_display.index)))
ax.set_yticklabels(S_display.index, fontsize=9)

ax.set_title('Formula-Symptom 행렬 S (가중치)', fontsize=13, pad=12)
ax.set_xlabel('증상')
ax.set_ylabel('처방')

plt.tight_layout()
plt.savefig('../data/S_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: data/S_heatmap.png')

## 4. 처방 간 코사인 유사도

In [ ]:
sim_F = cosine_similarity_matrix(F)
sim_F_display = sim_F.rename(index=formula_names, columns=formula_names)

print('=== 처방 간 유사도 (본초 구성 기반) ===')
sim_F_display.round(3)

In [ ]:
def plot_similarity_heatmap(sim_df, title):
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(sim_df.values, vmin=0, vmax=1, cmap='RdYlGn')
    plt.colorbar(im, ax=ax, label='코사인 유사도')

    labels = list(sim_df.index)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)

    # 셀 값 표시
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = sim_df.values[i, j]
            color = 'white' if val > 0.7 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=color)

    ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.show()

plot_similarity_heatmap(sim_F_display, '처방 간 코사인 유사도 (본초 구성 기반)')

In [ ]:
sim_S = cosine_similarity_matrix(S)
sim_S_display = sim_S.rename(index=formula_names, columns=formula_names)

print('=== 처방 간 유사도 (증상 기반) ===')
plot_similarity_heatmap(sim_S_display, '처방 간 코사인 유사도 (증상 기반)')
sim_S_display.round(3)

## 5. 증상 쿼리 → 처방 추천 테스트

환자 증상을 리스트로 입력하면 가장 유사한 처방을 추천합니다.

In [ ]:
name_idx = build_symptom_name_index(syndromes_df)
print('인식 가능한 증상 목록:')
print(list(name_idx.keys()))

In [ ]:
def recommend(symptoms: list, top_k: int = 5):
    """
    증상 이름 리스트를 받아 상위 top_k 처방을 반환.
    참고용 출력이며 임상 판단을 대체하지 않습니다.
    """
    missing = [s for s in symptoms if s not in name_idx]
    if missing:
        print(f'[경고] 데이터에 없는 증상: {missing}')

    q = pd.Series({name_idx[s]: 1.0 for s in symptoms if s in name_idx})
    if q.empty:
        print('유효한 증상이 없습니다.')
        return

    scores = query_similarity(q, S)
    print(f'\n[쿼리] {symptoms}')
    print(f'{'처방':<20} {'유사도':>8}  변증')
    print('-' * 50)
    for fid, score in scores.head(top_k).items():
        fname = formulas_df.loc[fid, 'name_kr']
        syn_ids = formulas_df.loc[fid, 'indications']['syndromes']
        syn_names = ', '.join(syndromes_df.loc[s, 'name_kr'] for s in syn_ids if s in syndromes_df.index)
        print(f'{fname:<20} {score:>8.4f}  {syn_names}')
    print('\n⚠️ 참고용 — 최종 처방은 임상의가 판단해야 합니다.')


# 테스트 1: 태양병 상한증 전형 증상
recommend(['오한', '발열', '무한', '두통', '신체통'])

In [ ]:
# 테스트 2: 태양병 중풍증 전형 증상
recommend(['발열', '오풍', '자한', '두통'])

In [ ]:
# 테스트 3: 소양병 증상
recommend(['왕래한열', '흉협고만', '구고', '인건'])

In [ ]:
# 테스트 4: 양명경증 4대증
recommend(['대열', '대한', '대갈', '맥홍대'])

In [ ]:
# 테스트 5: 소음병 한증
recommend(['사지궐냉', '오한권와', '맥미욕절'])

## 6. 처방 비교 — 마황탕 vs 계지탕

In [ ]:
def compare_formulas(fid_a: str, fid_b: str):
    """두 처방의 본초 구성과 코사인 유사도를 비교. 참고용."""
    fa = formulas_df.loc[fid_a]
    fb = formulas_df.loc[fid_b]

    herbs_a = {item['herb_id']: item for item in fa['composition']}
    herbs_b = {item['herb_id']: item for item in fb['composition']}

    common = set(herbs_a) & set(herbs_b)
    only_a = set(herbs_a) - set(herbs_b)
    only_b = set(herbs_b) - set(herbs_a)

    sim = float(sim_F.loc[fid_a, fid_b])

    print(f'=== {fa["name_kr"]} vs {fb["name_kr"]} ===')
    print(f'코사인 유사도 (본초 기반): {sim:.4f}')
    print()

    print(f'공통 본초 ({len(common)}종):')
    for hid in sorted(common):
        ha, hb = herbs_a[hid], herbs_b[hid]
        print(f'  {ha["name_kr"]:8s}  {fa["name_kr"]}: {ha["dose_g"]}g  |  {fb["name_kr"]}: {hb["dose_g"]}g')

    print(f'\n{fa["name_kr"]} 고유 ({len(only_a)}종):')
    for hid in sorted(only_a):
        h = herbs_a[hid]
        print(f'  {h["name_kr"]:8s}  {h["dose_g"]}g  [{h["role"]}]')

    print(f'\n{fb["name_kr"]} 고유 ({len(only_b)}종):')
    for hid in sorted(only_b):
        h = herbs_b[hid]
        print(f'  {h["name_kr"]:8s}  {h["dose_g"]}g  [{h["role"]}]')

    print('\n⚠️ 참고용 — 임상 적용은 전문가 판단이 필요합니다.')


compare_formulas('SHL_001', 'SHL_002')

In [ ]:
# 마황탕 vs 대청룡탕 (마황탕 + 석고)
compare_formulas('SHL_001', 'SHL_004')

---
## Phase 0 완료 체크리스트

- [x] `load_data.py` — JSON → DataFrame 로드 확인
- [x] `build_matrices.py` — F, S 행렬 생성 확인
- [x] F 행렬 히트맵 시각화
- [x] S 행렬 히트맵 시각화
- [x] 처방 간 유사도 확인 (본초 기반, 증상 기반)
- [x] 증상 쿼리 → 처방 추천 테스트
- [ ] **연구자 검증**: 시드 데이터 용량, 군신좌사 배정, 증상 가중치 검토 후 수정

다음 단계 → `src/engine.py` (Phase 1)